# 01 · Data quality and exploratory analysis

Treatment of the six data-quality issues identified in the proposal (Section IV-B), followed by exploratory analysis.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
coll = pd.read_parquet(PROC / "collisions.parquet")
wx = pd.read_parquet(PROC / "weather_hourly.parquet")
coll = coll[coll["year"].between(2014, 2025)]
print(f"{len(coll):,} collisions 2014-2025")

## Data-quality issues and their treatment

| # | Issue | Treatment |
|---|---|---|
| 1 | Missing locations | kept for city-wide analysis, excluded from spatial joins |
| 2 | Class imbalance; duplicates | class weights; models re-estimated without duplicates (notebook 07) |
| 3 | Locations snapped to road-network points | buffers instead of exact matches |
| 4 | No station records every variable | composite weather table |
| 5 | Weather in standard time, WYS/collisions in local time | weather converted to local time; transition hours flagged |
| 6 | COVID-19 structural break | COVID-19 indicator in every model |

In [ ]:
from src import collisions as C
dq = pd.DataFrame({
    "issue": ["1 missing coordinates", "2 severe share", "2 located duplicates", "3 distinct located points",
              "5 DST transition hours flagged", "4 hours with precipitation from fallback station"],
    "value": [f"{(~coll['has_location']).sum():,} ({100*(~coll['has_location']).mean():.1f}%)",
              f"{100*coll['severe'].mean():.1f}%",
              f"{coll['is_duplicate'].sum():,}",
              f"{coll.loc[coll['has_location'], ['lat','lon']].drop_duplicates().shape[0]:,} points for {coll['has_location'].sum():,} records",
              f"{wx['dst_transition'].sum()}",
              f"{wx['precip_filled'].sum():,}"]})
dq.to_csv(TAB / "data_quality.csv", index=False)
dq

## Annual collisions and the COVID-19 break

In [ ]:
ann = C.annual_counts(coll)
ann.to_csv(TAB / "annual_collisions.csv")
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(ann.index, ann["records"], color="#4C72B0", label="all collisions")
ax.bar(ann.index, ann["injury_or_fatal"], color="#DD8452", label="injury or fatal")
ax.axvspan(2019.5, 2021.5, color="grey", alpha=0.15, label="COVID-19 period")
ax.set_xlabel("year"); ax.set_ylabel("collisions"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Police-reported collisions in Toronto, 2014-2025")
fig.savefig(FIG / "fig_annual_collisions.png"); plt.show()

## Collisions per hour by weather condition

In [ ]:
counts = coll.groupby(["date", "hour"]).size().rename("collisions")
ch = wx.set_index(["date", "hour"]).join(counts).fillna({"collisions": 0}).reset_index()
by_w = ch.groupby("weather_cat")["collisions"].agg(["mean", "count"]).rename(columns={"mean": "collisions_per_hour", "count": "hours"})
by_w["vs_dry"] = by_w["collisions_per_hour"] / by_w.loc["dry_clear", "collisions_per_hour"]
by_w.to_csv(TAB / "collisions_per_hour_by_weather.csv")
display(by_w.round(3))

order = ["dry_clear", "rain", "snow", "low_visibility"]
by_hw = ch.groupby(["hour", "weather_cat"])["collisions"].mean().unstack()[order]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(order, by_w.loc[order, "collisions_per_hour"], color=["#8c8c8c", "#4C72B0", "#55A868", "#C44E52"])
axes[0].set_ylabel("mean collisions per city-hour"); axes[0].set_title("By weather (unadjusted)")
for c, col in zip(order, ["#8c8c8c", "#4C72B0", "#55A868", "#C44E52"]):
    axes[1].plot(by_hw.index, by_hw[c], label=c, color=col)
axes[1].set_xlabel("hour of day"); axes[1].set_title("By hour of day and weather"); axes[1].legend(frameon=False, fontsize=8)
fig.savefig(FIG / "fig_collisions_by_weather.png"); plt.show()

## Severity by road user and by weather

In [ ]:
users = ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"]
sev_user = pd.Series({u: coll.loc[coll[u], "severe"].mean() for u in users}, name="severe_share")
cw = coll.merge(wx[["date", "hour", "weather_cat"]], on=["date", "hour"])
sev_w = cw.groupby("weather_cat")["severe"].mean().rename("severe_share")
display(pd.concat([sev_user, sev_w]).to_frame().round(3))

## Weather station coverage and composite sources

In [ ]:
src_tab = pd.DataFrame({c: wx[f"{c}_src"].value_counts(normalize=True).round(4)
                        for c in ["precip_mm", "visibility_km", "temp_c", "weather_text"]})
src_tab

## WYS speed data and Bike Share exposure (overview)

In [ ]:
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
from pyspark.sql import functions as F
w = spark.read.parquet(str(PROC / "wys_raw"))
print("WYS rows 2023-2025:", f"{w.count():,}", "| signs:", w.select("sign_id").distinct().count())
bins = (w.groupBy("speed_bin").agg(F.sum("volume").alias("vehicles")).toPandas())
bins["lo"] = bins["speed_bin"].str.extract(r"\[(\d+),").astype(int)
bins = bins.sort_values("lo")
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bins["lo"] + 2.5, bins["vehicles"] / bins["vehicles"].sum(), width=4.5, color="#4C72B0")
ax.set_xlabel("speed (km/h)"); ax.set_ylabel("share of vehicles"); ax.set_title("Speeds recorded by WYS signs, 2023-2025")
fig.savefig(FIG / "fig_wys_speed_distribution.png"); plt.show()
spark.stop()

**Observations** (to be written from the output above): direction and size of the unadjusted weather differences, the COVID-19 drop, the very different severity shares by road user, and the station coverage that motivated the composite weather table.